In [1]:
# ===============================
# STEP 1 – IMPORT LIBRARIES
# ===============================

import pandas as pd
import mysql.connector
import time


# ===============================
# STEP 2 – LOG FUNCTION
# ===============================

def log(message):
    print(f"\n[LOG]: {message}")


# ===============================
# STEP 3 – LOAD DATA
# ===============================

def load_data():

    log("Loading CSV Data")

    df = pd.read_csv("../data/faculty_workload_batch.csv")

    print("\nLoaded Data:\n")
    print(df)

    return df


# ===============================
# STEP 4 – VALIDATE DATA
# ===============================

def validate_data(df):

    log("Validating Data")

    print("\nNull Values:\n")
    print(df.isnull().sum())

    print("\nDuplicate Rows:", df.duplicated().sum())

    return df


# ===============================
# STEP 5 – CLEAN DATA
# ===============================

def clean_data(df):

    log("Cleaning Data")

    # Fix negative values
    df["hours_assigned"] = df["hours_assigned"].abs()

    # Fill missing values
    df["hours_assigned"] = df["hours_assigned"].fillna(0)

    # Remove duplicates
    df = df.drop_duplicates()

    print("\nCleaned Data:\n")
    print(df)

    return df


# ===============================
# STEP 6 – INSERT INTO MYSQL
# ===============================

def insert_data(df):

    log("Connecting to MySQL")

    connection = mysql.connector.connect(
        host="localhost",
        user="root",
        password="your_password",
        database="faculty_workload_db"
    )

    cursor = connection.cursor()

    log("Inserting Data into workload table")

    for i, row in df.iterrows():

        cursor.execute("""

            INSERT INTO workload
            (workload_id,
             faculty_id,
             subject_id,
             hours_assigned,
             semester)

            VALUES (%s,%s,%s,%s,%s)

        """, (

            200 + i,
            int(row["faculty_id"]),
            300 + i,
            int(row["hours_assigned"]),
            row["semester"]

        ))

    connection.commit()

    print("\nRows Inserted:", len(df))

    cursor.close()
    connection.close()

    log("MySQL Insert Completed")


# ===============================
# STEP 7 – SAVE LOG FILE
# ===============================

def save_log():

    log("Saving ingestion log")

    with open("../logs/ingestion_log.txt", "a") as f:

        f.write(
            f"Ingestion completed at {time.ctime()}\n"
        )


# ===============================
# STEP 8 – PIPELINE FUNCTION
# ===============================

def pipeline():

    print("\n===== BATCH INGESTION PIPELINE STARTED =====")

    start = time.time()

    try:

        df = load_data()

        df = validate_data(df)

        df = clean_data(df)

        insert_data(df)

        save_log()

        print("\n===== PIPELINE COMPLETED SUCCESSFULLY =====")

    except Exception as e:

        print("\nPipeline Failed:", e)

    end = time.time()

    print("\nExecution Time:", end - start)


# ===============================
# STEP 9 – RUN PIPELINE
# ===============================

pipeline()


===== BATCH INGESTION PIPELINE STARTED =====

[LOG]: Loading CSV Data

Loaded Data:

   faculty_id faculty_name department          subject  hours_assigned  \
0           9   Anil Verma        CSE               ML             4.0   
1          10   Pooja Nair        ECE  Microprocessors             3.0   
2          11    Rahul Sen         IT  Cloud Computing             5.0   
3          12  Simran Kaur         ME    Manufacturing             2.0   
4          13  Vikram Shah        CSE         Big Data            -4.0   
5          14   Nisha Iyer        ECE             VLSI             NaN   

  semester  
0     Sem1  
1     Sem1  
2     Sem2  
3     Sem2  
4     Sem1  
5     Sem1  

[LOG]: Validating Data

Null Values:

faculty_id        0
faculty_name      0
department        0
subject           0
hours_assigned    1
semester          0
dtype: int64

Duplicate Rows: 0

[LOG]: Cleaning Data

Cleaned Data:

   faculty_id faculty_name department          subject  hours_assigned  \
0